## Model Comparison with Simulation-Based Inference and Harmonic

In this notebook, we have implemented the computation of the model evidence using the [Harmonic](https://github.com/astro-informatics/harmonic) package. This assumes that you have already completed an **SNLE** training and subsequently run the script `mlpoppyns/learning/utils/posterior_sampler_mcmc_sbi.py` using the trained model. This last script draws samples from the posterior distribution of the observed data and computes the corresponding log-probability values using the trained model. We have not included this step in the notebook because it is time-consuming, and performing the computation here with the required **MCMC sampler** would not be feasible due to limited computational resources. Note that sampling is performed multiple times, creating a chain, because this is the required input for Harmonic, a chain of posterior samples, similar to the output of an MCMC sampler.  

To run a SNLE training, use the following command:
```
python  ../../mlpoppyns/learning/sbi_train.py --c config_sbi.json 
```
Make sure that in the `config_sbi.json` file, the "trainer" section has the "type" set to "snle".


In [ ]:
import json
import os
import pathlib
import pickle

import corner
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sbi import utils

import time
import harmonic as hm

from sbi.inference.potentials import likelihood_estimator_based_potential
import mlpoppyns.learning.utils.sbi_utils as ut
import logging

Creating a basic logger to use the functions in `sbi_utils.py`.

In [ ]:
logger = logging.getLogger("logger")
logger.setLevel(logging.INFO)

# Add a console handler (optional)
handler = logging.StreamHandler()
formatter = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s')
handler.setFormatter(formatter)
logger.addHandler(handler)

Load the dataset statistics to rescale the parameters to their physical ranges.

In [ ]:
def import_statistics(stats_path: str):
    """
    Extracting the mean and standard deviation for all the parameters in the `stats_path` file.
    Args:
        stats_path (str): Path to the file where the statistics are saved.
    Returns:
        (torch.tensor, torch.tensor): Mean and standard deviation for the parameters in the `stats_path` file.
    """
    std_list = []
    mean_list = []
    max_list = []
    min_list = []
    with open(stats_path, "r") as json_file:
        data = json.load(json_file)
    for key, value in data.items():
        std_list.append(value["std"])
        mean_list.append(value["mean"])
        max_list.append(value["max"])
        min_list.append(value["min"])
    mean = np.array(mean_list)
    std = np.array(std_list)
    max_list = np.array(max_list)
    min_list = np.array(min_list)
    return mean, std, max_list, min_list

In [ ]:
config_path = '../../mlpoppyns/learning/config_sbi.json'
with open(config_path, 'r') as file:
    config = json.load(file)

stats_path = config["training_data_loader"]["statistic_path"]
par_mean, par_std, par_max, par_min = import_statistics(stats_path)

Loading the posterior samples and their log-probability.

In [ ]:
posterior_obs_chain = torch.load('/Users/celsapardoaraujo/Downloads/posterior_obs_chain_2.pt')
log_prob_chain = torch.load('/Users/celsapardoaraujo/Downloads/log_prob_chain_2.pt')

Plot one of the posterior samples of the chain.

In [ ]:
observed_samples = torch.tensor(posterior_obs_chain[2])

In [ ]:
# If the parameters were normalized or standardized rescale quantities to their physical ranges.
if config["training_data_loader"]["normalize"]:
    observed_samples = observed_samples * (par_max - par_min) + par_min

elif config["training_data_loader"]["standardize"]:
    observed_samples = observed_samples * par_std + par_mean

# Saving the best estimated parameters and the 95% CI into the log.txt file.
quantile = np.quantile(observed_samples, [0.025, 0.5, 0.975], axis=0)

range_param = [[par_min[v], par_max[v]] for v in range(len(par_max))]

param_median = quantile[1, :]

parameter_labels = [
    r"$\mu_{\log B}$",
    r"$\sigma_{\log B}$",
    r"$\mu_{\log P}$",
    r"$\sigma_{\log P}$",
    r"$a_{\rm late}$",
    r"$\mu_{\log L_0}$",
    r"$\alpha$"
]

figure = corner.corner(
    observed_samples.detach().cpu().numpy(),
    bins=32,
    labels=parameter_labels,
    range=range_param,
    quantiles=[0.025, 0.5, 0.975],
    levels=(
        1 - np.exp(-0.5),
        1 - np.exp(-2),
        1 - np.exp(-9.0 / 2.0),
    ),  # 1, 2 and 3 sigma levels
    show_titles=True,
    title_kwargs={"fontsize": 12},
)
corner.overplot_lines(figure, param_median, color="tab:red")

corner.overplot_points(
    figure,
    param_median[None],
    marker="s",
    color="tab:red",
)

Initialize the Chain object for Harmonic.

In [ ]:
ndim = posterior_obs_chain[0].shape[1]
chains = hm.Chains(ndim)
chains.add_chains_3d(posterior_obs_chain, log_prob_chain)

Splitting the data into a training dataset and an inference dataset. We use the former to train the normalizing flow to learn the harmonic mean estimator, and the latter to evaluate the trained normalizing flow and compute the evidence.

In [ ]:
chains_train, chains_infer = hm.utils.split_data(chains, training_proportion=0.5)

# Initialize and training the normalizing flow.

The temperature parameter refers to the standard deviation of the base distribution (a normal distribution) used in the normalizing flow. During sampling, points are first drawn from this base distribution and then transformed through the flow to match the approximated posterior. To ensure that the flow is concentrated on the high-density regions of the posterior, the temperature should be set to a value less than one. A commonly recommended value is 0.8.

In [ ]:
temperature = 0.6
model = hm.model.RealNVPModel(ndim, standardize=True, temperature=temperature)
epochs_num = 20

In [ ]:
model.fit(chains_train.samples, epochs=epochs_num, verbose= True)

Plot the concentrated flow (target distribution) and the posterior distribution. Remember that the former should be concentrated within the posterior distribution.

In [ ]:
samples = observed_samples.reshape((-1, ndim))
samp_num = observed_samples.shape[0]
flow_samples = model.sample(samp_num)


In [ ]:
fig = hm.utils.plot_getdist_compare(posterior_obs_chain[0], flow_samples)

Computing the log(1/z) and its error.

In [ ]:
# Instantiate harmonic's evidence class
ev = hm.Evidence(chains_infer.nchains, model)

# Pass the evidence class the inference chains and compute the evidence!
ev.add_chains(chains_infer)
ln_inv_evidence = ev.ln_evidence_inv
err_ln_inv_evidence = ev.compute_ln_inv_evidence_errors()
print(ln_inv_evidence,err_ln_inv_evidence)

In [ ]:
print('evidence:',10**(-ln_inv_evidence))